# Pharmaceutical Sales Dataset: Preprocessing & Feature Engineering
## Machine Learning Pipeline for Demand Forecasting

### 1. Preprocessing Objectives
1. Verify chronological continuity (2,106 days from 2014-01-02 to 2019-10-08).
2. Construct lag features: **Lag 1**, **Lag 7**, **Lag 14**, **Lag 30**.
3. Construct rolling window metrics: **Rolling Mean 7d, 14d, 30d**, **Rolling Std 7d, 30d**.
4. Extract calendar seasonality features: **Day of Week**, **Month**, **Year**, **Weekend Indicator**, **Quarter**.
5. Structure time-series dataset ready for chronological train-test splitting (strictly avoiding data leakage).


In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../../dataset/raw/salesdaily.csv')
df['datum'] = pd.to_datetime(df['datum'])
df = df.sort_values('datum').reset_index(drop=True)
atc_cols = ['M01AB', 'M01AE', 'N02BA', 'N02BE', 'N05B', 'N05C', 'R03', 'R06']

# Set date as index
df_ts = df.set_index('datum')
print(f'Continuous time-series loaded: {df_ts.shape[0]} daily records.')
df_ts[atc_cols].head()

Continuous time-series loaded: 2106 daily records.


,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
datum,,,,,,,,
2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0
2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0
2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0
2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0
2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0


### 2. Calendar & Temporal Feature Engineering
Capturing periodic customer behavior patterns, weekly refill cycles, and seasonal flu/allergy months.


In [2]:
features_df = pd.DataFrame(index=df_ts.index)
features_df['day_of_week'] = df_ts.index.dayofweek
features_df['day_of_month'] = df_ts.index.day
features_df['month'] = df_ts.index.month
features_df['year'] = df_ts.index.year
features_df['quarter'] = df_ts.index.quarter
features_df['is_weekend'] = df_ts.index.dayofweek.isin([5, 6]).astype(int)
features_df['is_month_end'] = df_ts.index.is_month_end.astype(int)
features_df.head()

,day_of_week,day_of_month,month,year,quarter,is_weekend,is_month_end
datum,,,,,,,
2014-01-02,3,2,1,2014,1,0,0
2014-01-03,4,3,1,2014,1,0,0
2014-01-04,5,4,1,2014,1,1,0
2014-01-05,6,5,1,2014,1,1,0
2014-01-06,0,6,1,2014,1,0,0


### 3. Autoregressive Lags & Rolling Window Features
Constructing multi-horizon predictive features for each therapeutic class:
- `lag_1`: Previous day's demand (short-term momentum)
- `lag_7`: Same day of previous week (captures 7-day cyclicality)
- `lag_14`: Two weeks prior
- `lag_30`: Monthly prior
- `rolling_mean_7`: Weekly smoothed velocity
- `rolling_mean_14`: Bi-weekly velocity
- `rolling_mean_30`: Monthly baseline velocity
- `rolling_std_7` & `rolling_std_30`: Local demand volatility


In [3]:
engineered_features = features_df.copy()

for col in atc_cols:
    series = df_ts[col]
    engineered_features[f'{col}_target'] = series
    # Lags
    engineered_features[f'{col}_lag_1'] = series.shift(1)
    engineered_features[f'{col}_lag_7'] = series.shift(7)
    engineered_features[f'{col}_lag_14'] = series.shift(14)
    engineered_features[f'{col}_lag_30'] = series.shift(30)
    # Rolling features (shift by 1 to prevent target leakage)
    shifted_series = series.shift(1)
    engineered_features[f'{col}_rolling_mean_7'] = shifted_series.rolling(7, min_periods=3).mean()
    engineered_features[f'{col}_rolling_mean_14'] = shifted_series.rolling(14, min_periods=7).mean()
    engineered_features[f'{col}_rolling_mean_30'] = shifted_series.rolling(30, min_periods=14).mean()
    engineered_features[f'{col}_rolling_std_7'] = shifted_series.rolling(7, min_periods=3).std()
    engineered_features[f'{col}_rolling_std_30'] = shifted_series.rolling(30, min_periods=14).std()

# Drop rows with NaN from initial 30-day lag warm-up
engineered_clean = engineered_features.dropna().copy()
print(f'Final Engineered Feature Set: {engineered_clean.shape[0]} rows, {engineered_clean.shape[1]} columns')
engineered_clean.head()

Final Engineered Feature Set: 2076 rows, 87 columns


,day_of_week,day_of_month,month,year,quarter,is_weekend,is_month_end,M01AB_target,M01AB_lag_1,M01AB_lag_7,...,R06_target,R06_lag_1,R06_lag_7,R06_lag_14,R06_lag_30,R06_rolling_mean_7,R06_rolling_mean_14,R06_rolling_mean_30,R06_rolling_std_7,R06_rolling_std_30
datum,,,,,,,,,,,,,,,,,,,,,
2014-02-01,5,1,2,2014,1,1,0,4.33,1.00,6.00,...,0.0,0.0,1.0,2.0,2.0,1.857143,1.785714,1.606667,1.772811,1.320902
2014-02-02,6,2,2,2014,1,1,0,7.00,4.33,4.33,...,0.0,0.0,0.0,1.0,4.0,1.714286,1.642857,1.540000,1.889822,1.350504
2014-02-03,0,3,2,2014,1,0,0,5.00,7.00,4.34,...,0.0,0.0,3.0,3.0,1.0,1.714286,1.571429,1.406667,1.889822,1.295598
2014-02-04,1,4,2,2014,1,0,0,1.33,5.00,6.00,...,2.0,0.0,5.0,1.0,0.0,1.285714,1.357143,1.373333,1.889822,1.319073
2014-02-05,2,5,2,2014,1,0,0,3.00,1.33,5.33,...,1.0,2.0,2.0,2.0,2.0,0.857143,1.428571,1.440000,1.069045,1.297637


### 4. Data Export and Catalog Mapping
Exporting cleaned datasets to `dataset/processed/` for model training and API services.


In [4]:
os.makedirs('../../dataset/processed', exist_ok=True)
df.to_csv('../../dataset/processed/cleaned_pharma_daily.csv', index=False)
engineered_clean.to_csv('../../dataset/processed/engineered_features.csv')
print('Saved cleaned_pharma_daily.csv and engineered_features.csv successfully!')

Saved cleaned_pharma_daily.csv and engineered_features.csv successfully!
